In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/mumbai/bus")

routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)
print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)
print("\nStop times columns:", stop_times.columns.tolist())

Routes: (1609, 5)
Trips: (99080, 5)
Stops: (10279, 4)
Stop times: (2655026, 6)

Stop times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'timepoint']


In [2]:
trip_counts = trips.groupby("route_id")["trip_id"].nunique().reset_index(name="total_trips")

trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

stop_counts = trip_stops.groupby("route_id")["stop_id"].nunique().reset_index(name="total_stops")

route_features = routes.merge(trip_counts, on="route_id", how="left")
route_features = route_features.merge(stop_counts, on="route_id", how="left")

print(route_features.head())

    route_id agency_id route_short_name  \
0  KDMT_5447      KDMT               01   
1  KDMT_5443      KDMT               02   
2  KDMT_5499      KDMT               03   
3  KDMT_5500      KDMT               03   
4  KDMT_5793      KDMT               03   

                                     route_long_name route_type  total_trips  \
0  Kalyan station To Kalyan station Via Durgadi (...          3          NaN   
1  Kalyan station To Kalyan station Via Birla Col...          3          1.0   
2                 Sai Chowk to Kalyan Station (DOWN)          3          NaN   
3                   Kalyan Station to Sai Chowk (UP)          3          NaN   
4                            Depot To Sai Chowk (UP)          3          NaN   

   total_stops  
0          NaN  
1         26.0  
2          NaN  
3          NaN  
4          NaN  


In [3]:
def gtfs_hour(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h + m / 60 + s / 3600
    except:
        return None

stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_hour)

def time_period(h):
    if pd.isna(h):
        return "Unknown"
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

stop_times["arrival_period"] = stop_times["arrival_hour"].apply(time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(time_period)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(stop_times[["arrival_hour", "departure_hour", "departure_period"]].head())
print("\nHourly trips:")
print(hourly_trips.head())

   arrival_hour  departure_hour departure_period
0      0.666667        0.666667            Night
1      0.733333        0.733333            Night
2      0.766667        0.766667            Night
3      0.800000        0.800000            Night
4      0.833333        0.833333            Night

Hourly trips:
    route_id  departure_hour  trip_count
0  BEST_1169        0.000000           1
1  BEST_1169        0.116667           1
2  BEST_1169        0.500000           1
3  BEST_1169        0.666667           1
4  BEST_1169        0.750000           1


In [4]:
output_path = Path("../data/processed/mumbai")
output_path.mkdir(parents=True, exist_ok=True)

route_features.to_csv(output_path / "route_profile.csv", index=False)
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Mumbai CSV files saved successfully!")

print("\nMissing values:\n", route_features.isnull().sum())
print("\nDuplicate route IDs:", route_features["route_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())
print("Invalid arrival hours:", stop_times["arrival_hour"].isna().sum())
print("Invalid departure hours:", stop_times["departure_hour"].isna().sum())

Mumbai CSV files saved successfully!

Missing values:
 route_id              0
agency_id             0
route_short_name      0
route_long_name       0
route_type            0
total_trips         494
total_stops         494
dtype: int64

Duplicate route IDs: 0
Duplicate trip IDs: 0
Invalid arrival hours: 0
Invalid departure hours: 0


In [5]:
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

output_path = Path("../data/processed/mumbai")
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Mumbai updated successfully!")

Mumbai updated successfully!
